# RAG with Pydantic AI

Retrieval-Augmented Generation (RAG) combines vector search with LLM generation. Pydantic AI's dependency injection pattern makes it easy to wire up a vector database as a tool.

```mermaid
flowchart LR
    User([User]) -->|Question| Agent
    Agent -->|Search query| Tool["Retrieve Tool"]
    Tool -->|Query| VDB[("ChromaDB")]
    VDB -->|Relevant docs| Tool
    Tool -->|Context| Agent
    Agent -->|Answer + Citations| User
```

Key Pydantic AI patterns used:
- **`deps_type`**: Inject dependencies (vector DB, embedding function) into the agent
- **`RunContext[Deps]`**: Access dependencies inside tools
- **`@agent.tool`**: Register retrieval as a tool the LLM can call

Reference: https://ai.pydantic.dev/examples/rag/

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import os
from dataclasses import dataclass

import chromadb
import logfire
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic_ai import Agent, RunContext

load_dotenv()

logfire.configure()
logfire.instrument_pydantic_ai()

## Basic RAG (no vector search)

First, a simple approach: load the entire PDF into context and let the LLM answer questions.

In [ ]:
file_path = "assets/bbva.pdf"
loader = PyPDFLoader(file_path)
pages = []

for page in loader.lazy_load():
    pages.append(page)

In [ ]:
context = ""
for i, page in enumerate(pages):
    context += f"--- PAGE {i + 1} ---\n{page.page_content}\n\n"

agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are a helpful assistant that can answer questions about the provided context.\n\n"
        "Please cite the page number used to answer the question. "
        'Write the page number in the format "Page X" at the end of your answer.\n\n'
        "If the answer is not found in the context, please say so."
    ),
)

question = "What is the main idea of the document?"
response = agent.run_sync(
    f"Please answer the following question based on the context provided:\n\n"
    f"Question: {question}\n\nDocuments:\n{context}"
)
print(response.output)

In [ ]:
question = "What are the daily transaction limits?"
response = agent.run_sync(
    f"Please answer the following question based on the context provided:\n\n"
    f"Question: {question}\n\nDocuments:\n{context}"
)
print(response.output)

## RAG with vector search

Now we'll use ChromaDB for vector search. The key Pydantic AI pattern here is **dependency injection**: we define a `Deps` dataclass that holds the ChromaDB collection, then inject it into the agent's tools via `RunContext`.

```mermaid
flowchart TB
    subgraph Setup
        PDF["PDF"] --> Split["Text Splitter"]
        Split --> Embed["OpenAI Embeddings"]
        Embed --> Store[("ChromaDB")]
    end

    subgraph Query
        Q["Question"] --> Agent
        Agent -->|"Calls retrieve tool"| Retrieve["retrieve()"]
        Retrieve -->|"RunContext[Deps]"| Store
        Store -->|"Top-k docs"| Retrieve
        Retrieve -->|"Context"| Agent
        Agent --> Answer["Answer"]
    end
```

### Create and populate the vector database

In [ ]:
openai_ef = OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"), model_name="text-embedding-3-small"
)
vector_db = chromadb.PersistentClient()

try:
    vector_db.delete_collection("bbva")
except Exception:
    pass

collection = vector_db.create_collection("bbva", embedding_function=openai_ef)

### Split and index documents

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(pages)

collection.add(
    documents=[split.page_content for split in all_splits],
    metadatas=[split.metadata for split in all_splits],
    ids=[str(i) for i in range(len(all_splits))],
)

print(f"Indexed {len(all_splits)} chunks")

### Query the database

In [ ]:
collection.query(
    query_texts=["What are the daily transaction limits?", "Is there a monthly limit?"],
    n_results=3,
)

### Traditional RAG: Retrieve then generate

In this approach, we first retrieve relevant documents from the vector database, then pass them as context in the prompt. The LLM doesn't call any tools — retrieval happens before the LLM is invoked.

```mermaid
flowchart LR
    Q["Question"] --> Retrieve["Vector Search"]
    Retrieve --> Store[("ChromaDB")]
    Store --> Context["Retrieved Docs"]
    Context --> Agent["Agent (no tools)"]
    Agent --> Answer["Answer"]
```

In [ ]:
question = "What are the daily transaction limits?"

results = collection.query(query_texts=[question], n_results=3)
retrieved_context = "\n\n".join(
    f"--- PAGE {meta['page'] + 1} ---\n{doc}"
    for doc, meta in zip(results["documents"][0], results["metadatas"][0])
)

traditional_rag_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are a helpful assistant that can answer questions about the provided context.\n\n"
        "Please cite the page number used to answer the question. "
        'Write the page number in the format "Page X" at the end of your answer.\n\n'
        "If the answer is not found in the context, please say so."
    ),
)

response = traditional_rag_agent.run_sync(
    f"Please answer the following question based on the context provided:\n\n"
    f"Question: {question}\n\nDocuments:\n{retrieved_context}"
)
print(response.output)

### Agentic RAG: Retrieval as a tool

In this approach, the LLM decides when and what to retrieve by calling a tool. Following the [Pydantic AI RAG example](https://ai.pydantic.dev/examples/rag/), we define a `Deps` dataclass to hold the ChromaDB collection and inject it into the agent's retrieval tool via `RunContext[Deps]`.

In [ ]:
@dataclass
class Deps:
    collection: chromadb.Collection


rag_agent = Agent(
    "openai:gpt-5-nano",
    deps_type=Deps,
    system_prompt=(
        "You are a helpful assistant that can answer questions about a document.\n\n"
        "Use the retrieve tool to search for relevant information before answering.\n\n"
        "Please cite the page number used to answer the question. "
        'Write the page number in the format "Page X" at the end of your answer.\n\n'
        "If the answer is not found in the retrieved context, please say so."
    ),
)


@rag_agent.tool
def retrieve(context: RunContext[Deps], search_query: str) -> str:
    """Retrieve relevant document sections based on a search query.

    Args:
        context: The call context.
        search_query: The search query.
    """
    results = context.deps.collection.query(
        query_texts=[search_query],
        n_results=3,
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    return "\n\n".join(
        f"--- PAGE {meta['page']} ---\n{doc}" for doc, meta in zip(documents, metadatas)
    )

### Run the RAG agent

Pass the dependencies at runtime via `deps=`.

In [ ]:
deps = Deps(collection=collection)

response = rag_agent.run_sync(
    "What are the customer service channels?",
    deps=deps,
)
print(response.output)

In [ ]:
response = rag_agent.run_sync(
    "What are the daily transaction limits?",
    deps=deps,
)
print(response.output)

In [ ]:
response = rag_agent.run_sync(
    "How many cards can I have?",
    deps=deps,
)
print(response.output)

# Exercise

Download this book and create a vector database with it: https://github.com/mlschmitt/classic-books-markdown/blob/main/Friedrich%20Nietzsche/Beyond%20Good%20and%20Evil.md

Build a RAG agent that can answer questions about the book using the Pydantic AI dependency injection pattern.

In [ ]:
book_path = "assets/book.md"

with open(book_path, "r") as f:
    book_text = f.read()

book_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
)
book_chunks = book_splitter.split_text(book_text)

try:
    vector_db.delete_collection("beyond_good_and_evil")
except Exception:
    pass

book_collection = vector_db.create_collection(
    "beyond_good_and_evil",
    embedding_function=openai_ef,
)

book_collection.add(
    documents=book_chunks,
    metadatas=[{"chunk": i + 1} for i in range(len(book_chunks))],
    ids=[f"chunk-{i}" for i in range(len(book_chunks))],
)

print(f"Indexed {len(book_chunks)} chunks from {book_path}")

In [ ]:
@dataclass
class BookDeps:
    collection: chromadb.Collection


book_agent = Agent(
    "openai:gpt-5-nano",
    deps_type=BookDeps,
    system_prompt=(
        "You answer questions about the book 'Beyond Good and Evil'. "
        "Always use the retrieval tool before answering, and cite the chunk "
        "you used in the format 'Chunk X'."
    ),
)


@book_agent.tool
def retrieve_book_passages(ctx: RunContext[BookDeps], search_query: str) -> str:
    """Retrieve relevant passages from the book."""
    results = ctx.deps.collection.query(
        query_texts=[search_query],
        n_results=4,
    )
    return "\n\n".join(
        f"--- CHUNK {meta['chunk']} ---\n{doc}"
        for doc, meta in zip(
            results["documents"][0],
            results["metadatas"][0],
        )
    )


book_response = book_agent.run_sync(
    "What does Nietzsche suggest about the will to power?",
    deps=BookDeps(collection=book_collection),
)
print(book_response.output)